# Camera-ready revision experiments (DeSE 2026)
Self-contained: needs only `Training-data.csv` in the same folder. Run top to bottom. Results are saved as CSV files as it runs.

Addresses: repeated splits + mean/std (R1 #8, R2), paired test MLP vs GB (R2), equal tuning budget (R1 #5), train-only thresholds/scaling (R1 #4), architecture comparison for the depth claim (R1 #7), MLP convergence info (R1 #6).

In [ ]:
import os, tempfile
# Optional: keep joblib temp files on D: (same as your thesis notebook)
d_drive_temp = "D:\\Thesis_Temp_Storage"
if os.path.exists("D:\\"):
    os.makedirs(d_drive_temp, exist_ok=True)
    os.environ["JOBLIB_TEMP_FOLDER"] = d_drive_temp
    tempfile.tempdir = d_drive_temp
    print("Temp storage ->", d_drive_temp)

## CELL R1 — Setup & leak-free feature pipeline

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

SEED = 42
raw = pd.read_csv("Training-data.csv").dropna()

def row_features(d):
    """Row-wise features only (no statistics from other rows -> no leakage)."""
    d = d.copy()
    t = d["Tenure"] + 1
    d["Avg_Spend_Per_Tenure"] = np.log1p(d["Total Spend"] / t)
    d["Usage_Per_Tenure"]     = np.log1p(d["Usage Frequency"] / t)
    d["Support_Call_Rate"]    = d["Support Calls"] / t
    d["Payment_Delay_Rate"]   = np.log1p(d["Payment Delay"] / t)   # log applied ONCE
    d["Recency_Score"]        = 1 / (d["Last Interaction"] + 1)
    d["Engagement_Score"]     = d["Usage_Per_Tenure"] - d["Support_Call_Rate"] - d["Payment_Delay_Rate"]
    d["Age_Group"] = pd.cut(d["Age"], bins=[18,25,35,45,55,65,100],
                            labels=["18-25","26-35","36-45","46-55","56-65","65+"])
    d = pd.get_dummies(d, columns=["Gender","Subscription Type","Contract Length","Age_Group"],
                       drop_first=True)
    d = d.drop(columns=["CustomerID","Tenure","Usage Frequency","Total Spend",
                        "Last Interaction","Age"])
    # 'Support Calls' and 'Payment Delay' kept for now: HighRisk needs them, then drops them
    return d.replace([np.inf, -np.inf], np.nan).dropna().astype(float)

class HighRisk(BaseEstimator, TransformerMixin):
    """High_Risk flag with medians learned from the TRAINING fold only."""
    def fit(self, X, y=None):
        self.m_calls_ = X["Support Calls"].median()
        self.m_delay_ = X["Payment Delay"].median()
        return self
    def transform(self, X):
        X = X.copy()
        X["High_Risk"] = ((X["Support Calls"] > self.m_calls_) &
                          (X["Payment Delay"] > self.m_delay_)).astype(float)
        return X.drop(columns=["Support Calls", "Payment Delay"])

data = row_features(raw)
X_all = data.drop(columns="Churn")
y_all = data["Churn"].astype(int)

# Split off a small TUNING set that is never used in the final evaluation
X_eval, X_tune, y_eval, y_tune = train_test_split(
    X_all, y_all, test_size=40_000, stratify=y_all, random_state=SEED)
print("Evaluation set:", X_eval.shape, "| Tuning set:", X_tune.shape)

def make_pipe(model):
    # Same preprocessing for every model; fitted inside each training fold only
    return Pipeline([("risk", HighRisk()), ("scale", StandardScaler()), ("clf", model)])

## CELL R2 — Equal tuning budget for every model

In [ ]:
N_ITER, TUNE_CV = 10, 3          # identical budget: 10 random configs x 3-fold CV per model

search_spaces = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=SEED),
        {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]}),
    "Naive Bayes": (GaussianNB(),
        {"clf__var_smoothing": np.logspace(-11, -5, 10)}),
    "KNN": (KNeighborsClassifier(n_jobs=-1),
        {"clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
         "clf__weights": ["uniform", "distance"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
        {"clf__max_depth": [4, 6, 8, 10, 12, 16, None],
         "clf__min_samples_leaf": [1, 10, 50, 100]}),
    "Random Forest": (RandomForestClassifier(n_jobs=-1, random_state=SEED),
        {"clf__n_estimators": [100, 200, 300],
         "clf__max_depth": [8, 12, 16, None],
         "clf__min_samples_leaf": [1, 10, 50],
         "clf__max_features": ["sqrt", 0.5]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=SEED),
        {"clf__n_estimators": [100, 200, 300],
         "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
         "clf__max_depth": [3, 4, 5, 6],
         "clf__subsample": [0.8, 1.0]}),
    "MLP": (MLPClassifier(solver="adam", max_iter=300, early_stopping=True,
                          validation_fraction=0.1, n_iter_no_change=10, random_state=SEED),
        {"clf__hidden_layer_sizes": [(50,), (100,), (200,), (100, 50), (128, 64), (100, 50, 25)],
         "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
         "clf__learning_rate_init": [1e-3, 3e-3]}),
}

best_params, tuning_rows = {}, []
for name, (model, space) in search_spaces.items():
    t0 = time.time()
    n_iter = min(N_ITER, int(np.prod([len(v) for v in space.values()])))
    rs = RandomizedSearchCV(make_pipe(model), space, n_iter=n_iter, cv=TUNE_CV,
                            scoring="accuracy", random_state=SEED, n_jobs=1)
    rs.fit(X_tune, y_tune)
    best_params[name] = rs.best_params_
    tuning_rows.append({"Model": name, "Configs tried": n_iter,
                        "Best CV acc": round(rs.best_score_, 4),
                        "Best params": rs.best_params_,
                        "Minutes": round((time.time() - t0) / 60, 1)})
    if name == "MLP":            # keep every architecture's score for the depth question
        mlp_search_results = pd.DataFrame(rs.cv_results_)[
            ["param_clf__hidden_layer_sizes", "param_clf__alpha",
             "param_clf__learning_rate_init", "mean_test_score", "std_test_score"]
        ].sort_values("mean_test_score", ascending=False)
    print(f"{name:20s} best={rs.best_score_:.4f}  {rs.best_params_}")

tuning_table = pd.DataFrame(tuning_rows)
tuning_table.to_csv("rev_tuning_table.csv", index=False)
mlp_search_results.to_csv("rev_mlp_architectures.csv", index=False)
display(tuning_table)
display(mlp_search_results)

## CELL R3 — 5x2 repeated cross-validation

In [ ]:
# 5 repetitions x 2 folds = 10 train/test evaluations per model, same splits for every model
def tuned_model(name):
    model, _ = search_spaces[name]
    return make_pipe(clone(model)).set_params(**best_params[name])

rows, fold_acc = [], {name: [] for name in search_spaces}
for rep in range(5):
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=rep)
    for fold, (tr, te) in enumerate(skf.split(X_eval, y_eval)):
        Xtr, Xte = X_eval.iloc[tr], X_eval.iloc[te]
        ytr, yte = y_eval.iloc[tr], y_eval.iloc[te]
        for name in search_spaces:
            t0 = time.time()
            pipe = tuned_model(name).fit(Xtr, ytr)
            prob = pipe.predict_proba(Xte)[:, 1]
            pred = (prob >= 0.5).astype(int)
            acc = accuracy_score(yte, pred)
            fold_acc[name].append(acc)
            row = {"Model": name, "rep": rep, "fold": fold, "Accuracy": acc,
                   "Precision": precision_score(yte, pred), "Recall": recall_score(yte, pred),
                   "F1": f1_score(yte, pred), "ROC_AUC": roc_auc_score(yte, prob),
                   "fit_sec": time.time() - t0}
            if name == "MLP":
                row["mlp_epochs"] = pipe.named_steps["clf"].n_iter_
            rows.append(row)
            print(f"rep {rep} fold {fold} | {name:20s} acc={acc:.4f} ({row['fit_sec']:.0f}s)")
        pd.DataFrame(rows).to_csv("rev_5x2_raw_results.csv", index=False)  # saved as it goes

results = pd.DataFrame(rows)
summary = (results.groupby("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
           .agg(["mean", "std"]).sort_values(("Accuracy", "mean"), ascending=False))
summary.to_csv("rev_summary_mean_std.csv")
display(summary.round(4))

## CELL R4 — Paired statistical tests

In [ ]:
def paired_ttest_5x2cv(a, b):
    """Dietterich (1998) 5x2cv paired t-test. a, b = 10 fold scores in rep/fold order."""
    d = np.array(a).reshape(5, 2) - np.array(b).reshape(5, 2)
    s2 = ((d - d.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
    t = d[0, 0] / np.sqrt(s2.mean())
    p = 2 * stats.t.sf(abs(t), df=5)
    return t, p

best = summary.index[0]
test_rows = []
for other in summary.index[1:]:
    t, p = paired_ttest_5x2cv(fold_acc[best], fold_acc[other])
    w = stats.wilcoxon(fold_acc[best], fold_acc[other]).pvalue
    test_rows.append({"Comparison": f"{best} vs {other}",
                      "Mean diff (acc)": np.mean(fold_acc[best]) - np.mean(fold_acc[other]),
                      "5x2cv t": t, "5x2cv p": p, "Wilcoxon p": w})
tests = pd.DataFrame(test_rows)
tests.to_csv("rev_paired_tests.csv", index=False)
display(tests.round(4))

# The specific comparison Reviewer 2 asked for
t, p = paired_ttest_5x2cv(fold_acc["MLP"], fold_acc["Gradient Boosting"])
print(f"MLP vs Gradient Boosting: mean diff = "
      f"{np.mean(fold_acc['MLP']) - np.mean(fold_acc['Gradient Boosting']):.4f}, "
      f"5x2cv t = {t:.3f}, p = {p:.4f}")
print("MLP epochs to convergence (per fold):",
      results.loc[results.Model == "MLP", "mlp_epochs"].tolist())

## CELL R5 — (Optional) Keras ANN on the same splits

In [ ]:
# OPTIONAL — Keras ANN on the SAME 10 splits (needs TensorFlow). Skip if short on time.
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

keras_rows = []
for rep in range(5):
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=rep)
    for fold, (tr, te) in enumerate(skf.split(X_eval, y_eval)):
        tf.keras.utils.set_random_seed(SEED + rep)           # fixed seed per run
        prep = Pipeline([("risk", HighRisk()), ("scale", StandardScaler())])
        Xtr = prep.fit_transform(X_eval.iloc[tr]); ytr = y_eval.iloc[tr].values
        Xte = prep.transform(X_eval.iloc[te]);     yte = y_eval.iloc[te].values
        model = Sequential([Input(shape=(Xtr.shape[1],)),
                            Dense(128, activation="relu", kernel_initializer="glorot_uniform"),
                            Dropout(0.2),
                            Dense(64, activation="relu"),
                            Dense(1, activation="sigmoid")])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                      loss="binary_crossentropy", metrics=["accuracy"])
        es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        h = model.fit(Xtr, ytr, epochs=30, batch_size=1024, validation_split=0.1,
                      callbacks=[es], verbose=0)
        prob = model.predict(Xte, verbose=0).ravel(); pred = (prob >= 0.5).astype(int)
        keras_rows.append({"Model": "Keras ANN", "rep": rep, "fold": fold,
                           "Accuracy": accuracy_score(yte, pred),
                           "Precision": precision_score(yte, pred), "Recall": recall_score(yte, pred),
                           "F1": f1_score(yte, pred), "ROC_AUC": roc_auc_score(yte, prob),
                           "epochs_run": len(h.history["loss"])})
        print(f"rep {rep} fold {fold} | Keras acc={keras_rows[-1]['Accuracy']:.4f} "
              f"epochs={keras_rows[-1]['epochs_run']}")

keras_res = pd.DataFrame(keras_rows)
keras_res.to_csv("rev_keras_5x2.csv", index=False)
display(keras_res[["Accuracy","Precision","Recall","F1","ROC_AUC"]].agg(["mean","std"]).round(4))